In [8]:
import os 
import random
from alpha_blokus.neural_net import NeuralNet
from alpha_blokus.training.unlooped import GameDataset

import torch
from torch import nn

from hydra import compose, initialize

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name="unlooped_training", overrides=["networks=wide","game.moves_data_directory=../data/moves_20_4/"])
    print(cfg)

file_paths = []
for file_path in os.listdir(cfg["training"]["data_read_directory"]):
    if (
        file_path.endswith(".npz") and 
        file_path >= cfg["training"]["minimum_file"] and
        file_path <= cfg["training"]["maximum_file"]
    ):
        file_paths.append(os.path.join(cfg["training"]["data_read_directory"], file_path))

# Shuffle the files and split them into training and validation sets.
random.shuffle(file_paths)

# Load the model
model = NeuralNet(cfg["networks"]["main"], cfg)
model.to(cfg["training"]["device"])

train_dataset = GameDataset(cfg["training"]["device"], cfg, shuffle_files=False)
for file_path in file_paths:
    train_dataset.add_file(file_path)
    break
print("Loaded", train_dataset.total_samples, "samples for training")

boards, policies, values, valid_moves = train_dataset.get_batch(cfg["training"]["batch_size"])

# Forward pass
pred_values, pred_policy = model(boards)

# Get valid moves
pred_policy[~valid_moves] = -1e6

# Calculate losses
value_loss = nn.CrossEntropyLoss()(pred_values, values)
policy_loss = nn.CrossEntropyLoss()(pred_policy, policies)
loss = value_loss + cfg["training"]["policy_loss_weight"] * policy_loss

# Backward pass
loss.backward()

gradients = {
    'grad_norm/value_head': torch.cat([p.grad.view(-1) for p in model.value_head.parameters()]).norm(),
    'grad_norm/policy_head': torch.cat([p.grad.view(-1) for p in model.policy_head.parameters()]).norm(),
}

{'game': {'board_size': 20, 'num_moves': 30433, 'num_pieces': 21, 'num_piece_orientations': 91, 'moves_data_directory': '../../data/moves_20_4/'}, 'networks': {'main': {'main_body_channels': 128, 'residual_blocks': 10, 'value_head_channels': 24, 'value_head_flat_layer_width': 128, 'policy_head_channels': 128, 'policy_convolution_kernel': 3, 'new_model_check_interval': 120, 'batch_size': 128, 'model_read_path': 'models/', 'initialize_model_if_empty': False, 'log_gpu_evaluation': False}}, 'training': {'network_name': 'main', 'device': 'mps', 'batch_size': 128, 'policy_loss_weight': 0.158, 'learning_rate': 0.0005, 'sample_window': 100000, 'samples_per_generation': 10000, 'sampling_ratio': 2.0, 'minimum_window_size': 10000, 'new_data_check_interval': 60, 'data_read_directory': '/Users/shivamsarodia/Dev/BlokusBot/data/2024-12-30_23-23-24-rubefaction/games', 'model_write_directory': 'models/', 'validation_set_size': 0.1, 'num_epochs': 5, 'num_samples_between_test_evaluations': 200000.0, 'min

FileNotFoundError: [Errno 2] No such file or directory: '../../data/moves_20_4/'